Evaluating biosample host metadata labels in an attempt to capture all existing human labels using a ruleset.

In [89]:
import pandas as pd

In [90]:
file_location = "../data/Whole_BioSample.parquet"

df = pd.read_parquet(file_location)

In [91]:
hostnames = set(df['host_name'].to_list())

In [92]:
import re
from pathlib import Path

HUMAN_NAMES = ['homo', 'man', 'homo sapiens', 'human', 'patient', 'homo sapien', 'woman', '9606']

In [93]:

print(HUMAN_NAMES)
# Longest names first so multi-word names win over their prefixes ('homo sapiens' before 'homo')
WORD = '(?:' + '|'.join(re.escape(n) for n in sorted(HUMAN_NAMES, key=len, reverse=True)) + ')'

# "word" - the whole host name is a human name
accepted_re = re.compile(WORD, re.IGNORECASE)

# "--word", "word--", "--word--" - a human name fused to other non-whitespace characters
attached_re = re.compile(rf'\S+{WORD}|{WORD}\S+', re.IGNORECASE)

# "-- word", "word --", "-- word --" - a human name separated from other text by whitespace
spaced_re = re.compile(rf'\S+\s+{WORD}|{WORD}\s+\S+', re.IGNORECASE)

accepted, potential = set(), set()
potential_rows = {'host_name': [], 'human_name': []}
for name in hostnames:
    if not isinstance(name, str):
        continue
    if accepted_re.fullmatch(name.strip()):
        accepted.add(name)
    elif attached_re.search(name) or spaced_re.search(name):
        potential.add(name)
        # Most specific (longest) HUMAN_NAMES entry found in the host name
        potential_rows['host_name'].append(name)
        potential_rows['human_name'].append(max(accepted_re.findall(name), key=len).lower())

potential_human_labels = pd.DataFrame.from_dict(potential_rows)




DATA_DIR = Path('/home/clayg/humanpredict/data')
#(DATA_DIR / 'accepted_human_labels.txt').write_text('\n'.join(sorted(accepted)) + '\n', encoding='utf-8')
#(DATA_DIR / 'potential_human_labels.txt').write_text('\n'.join(sorted(potential)) + '\n', encoding='utf-8')

print(f"{len(accepted)} accepted, {len(potential)} potential, out of {len(hostnames)} host names")
print(HUMAN_NAMES)

['homo', 'man', 'homo sapiens', 'human', 'patient', 'homo sapien', 'woman', '9606']
12 accepted, 1290 potential, out of 19310 host names
['homo', 'man', 'homo sapiens', 'human', 'patient', 'homo sapien', 'woman', '9606']


Next is a fuzzy matching step which helps to identify typos and spelling errors. I will start at a minimum score of 90 (out of 100), and walk it back until lowest scores seem to be noise as opposed to valid errors

In [94]:
from rapidfuzz import fuzz

remaining_host_names = [host for host in hostnames if (host not in potential) & (host not in accepted)]
specific_human_names = HUMAN_NAMES[2:]
fuzzy_matches = {'host_name':[], 'human_name':[], 'fuzzy_score':[] }

for host in remaining_host_names:
    host = str(host)
    for name in specific_human_names:
        score = fuzz.ratio(host.lower(), name)

        if score > 65:
            fuzzy_matches['human_name'].append(name)
            fuzzy_matches['host_name'].append(host)
            fuzzy_matches['fuzzy_score'].append(score)
            break


fuzzy_matches_df = pd.DataFrame.from_dict(fuzzy_matches)

'''
DATA_DIR = Path('/home/clayg/humanpredict/data')
(DATA_DIR / 'fuzzy_human_labels.txt').write_text('\n'.join(sorted(fuzzy_matches)) + '\n', encoding='utf-8')
'''


"\nDATA_DIR = Path('/home/clayg/humanpredict/data')\n(DATA_DIR / 'fuzzy_human_labels.txt').write_text('\n'.join(sorted(fuzzy_matches)) + '\n', encoding='utf-8')\n"

In [95]:
fuzzy_matches_df['host_name']

0               4906
1       hono sapiens
2       hopo sapiens
3              plant
4       Gomo sapiens
5     Chelon saliens
6           Plantain
7        Hmo sapiens
8       Hopo sapiens
9          H sapiens
10           Humaine
11      home sapines
12             huamn
13           sapiens
14       Hot springs
15              Swan
16        H. sapiens
17             PLANT
18            wombat
19            Parent
20             Plant
21      Homp sapiens
22       Plastic net
23            Humain
24         Impatiens
25         h.sapiens
26     Hoimo sapiens
27       omo sapiens
28        h. sapiens
29        H.�sapiens
30      Hono sapiens
31           Sichuan
32      Himo sapiens
33       Hom sapiens
34          plantain
35        Attine ant
36             Huamn
Name: host_name, dtype: str

In [96]:
potential_human_labels = pd.concat(objs=[potential_human_labels, fuzzy_matches_df])
print(len(potential_human_labels))

1327


In [97]:
print(f"potential human labels dataframe contains columns: {potential_human_labels.columns.to_list()}")
print(f"fuzzy human labels dataframe contains columns: {fuzzy_matches_df.columns.to_list()}")

potential human labels dataframe contains columns: ['host_name', 'human_name', 'fuzzy_score']
fuzzy human labels dataframe contains columns: ['host_name', 'human_name', 'fuzzy_score']


In [33]:
potential_human_labels.to_csv("../data/potential_human_labels.csv")

Review of potential human labels was done externally. Now we'll create a new dataframe containing only valid human-labeled rows

In [99]:
potential_human_labels = pd.read_csv(r"../data/potential_human_labels_reviewed.csv")

valid_human_labels = set(potential_human_labels[potential_human_labels['clay_verification'] == True]['host_name']) | accepted
human_df = df[df['host_name'].str.lower().isin({l.lower() for l in valid_human_labels})]

In [100]:
human_df.head()

,biosample_accession,sample_taxon_id,sample_taxonomy_name,host_comments,host_name,isolation_comments,comments
1176,SAMN52580513,2890317,Yersinia alsatica,Not available,Homo sapiens,PHE | human,NaN
1178,SAMN52580515,666,Vibrio cholerae,Not available,Homo sapiens,PHE | human,NaN
4309,SAMN52583663,1314,Streptococcus pyogenes,NaN,Homo sapiens,blood,NaN
4310,SAMN52583664,1314,Streptococcus pyogenes,NaN,Homo sapiens,wound,NaN
4311,SAMN52583665,1314,Streptococcus pyogenes,NaN,Homo sapiens,wound,NaN


In [101]:
len(human_df)

1839776

In [102]:
#Assign 'category' columns as 1 for human
human_df['category'] = 1

### missing labels identification

In [103]:
import re
import numpy as np

missing_labels = {
    'missing', 'undefined', 'unknown', 'uncertain', 'undetermined', 'nan',
    'not collected', 'not provided', 'not available', 'not determined',
    'restricted access', 'unspecified', 'null', 'empty', 'blank', 'nd',
    'missing: data agreement established pre-2023',
    'missing: endangered species', 'missing: human-identifiable',
}

norm = (df['host_name'].astype('string').str.lower().str.strip()
        .str.strip('\'" ')
        .str.replace(r'(?<=[a-z])\d+$', '', regex=True)  # missing7 -> missing, but 9801 untouched
        .str.strip())


typo_pat = r'^(miss(ing)?[,:]?\s*)?not\s*c\w*ll?\w*ct\w*$|^(missing)+$|^[\W_]*$'

whole_df_uncertain_mask = (
    df['host_name'].isna()
    | norm.isin(missing_labels)
    | norm.str.fullmatch(typo_pat, na=False)
    | norm.str.startswith('not available: not collected', na=False)
)
missing_df = df[whole_df_uncertain_mask]


In [104]:
norm[norm.str.fullmatch(typo_pat, na=False)].value_counts()

host_name
missing           487760
not collected      14800
-                    339
- | -                144
not collect           90
#&                    47
not colllected         8
`                      1
not clooected          1
/                      1
not collectd           1
Name: count, dtype: int64[pyarrow]

In [105]:
len(missing_df)
missing_df['host_name'].unique()

<ArrowStringArray>
[                           nan,                      'missing',
                'not collected',                 'not provided',
                      'Missing',                            '-',
                      'Unknown',                'Not collected',
                'Not available',                      'unknown',
                'not available',                      'MISSING',
                 'undetermined',                 'Not Provided',
                  'not collect',                'Not Available',
                 'Not provided',            'restricted access',
 'not available: not collected',                'Not Collected',
                            '`',                        '- | -',
                'not clooected',                            '/',
               'not colllected',                         'NULL',
               'Not determined',                 'not collectd',
                         'null',                      'UNKNOWN',
      

In [113]:
missing_df['category'] = 0

In [114]:
human_biosamp = set(human_df['biosample_accession'].to_list())

missing_biosamp = set(missing_df['biosample_accession'].to_list())

not_human_missing = (~df['biosample_accession'].isin(human_biosamp))&(~df['biosample_accession'].isin(missing_biosamp))

not_human_missing_df = df[not_human_missing]

In [115]:
not_human_missing_df['category'] = -1

In [52]:
#checking overlap between key identifiers of each dataframe. There should be no overlap between any of them; 
# they are all mutually exlusive

human_not_human_overlap = sum(human_df['biosample_accession'].isin(not_human_missing_df['biosample_accession']))

human_uncertain_overlap = sum(human_df['biosample_accession'].isin(missing_df['biosample_accession']))

not_human_uncertain_overlap = sum(not_human_missing_df['biosample_accession'].isin(missing_df['biosample_accession']))

print(human_not_human_overlap, human_uncertain_overlap, not_human_uncertain_overlap)

0 0 0


In [116]:
final_df = pd.concat(objs=[human_df, missing_df, not_human_missing_df])

In [117]:
print(len(final_df))
print(len(df))

6060462
6060462


In [118]:
final_df = final_df.drop_duplicates(subset=["isolation_comments", "comments", 'category'])
len(final_df)

1511070

In [119]:
import numpy as np
final_df['category'] = final_df['category'].replace(np.nan, 0)

In [120]:
final_df['category'].unique()

array([ 1,  0, -1])

In [121]:
final_df.to_csv(r"/home/clayg/humanpredict/data/human_labeled.csv")

In [122]:
%reset -f

In [1]:
import pandas as pd

df = pd.read_csv("/home/clayg/humanpredict/data/human_labeled.csv")

df.to_parquet("/home/clayg/humanpredict/data/human_labeled.parquet")